# Measuring Inflation Shock Momentum — replication notebook

Self-contained replication of **Lansing & Shapiro (2026), FRBSF WP 2026-10**.
Builds the Inflation Shock Momentum (ISM) index from ~130 disaggregated PCE
categories and reproduces Figure 1 and the in-sample Table 1. Each section pairs
the paper's equation with the code. Validated at **0.99 correlation** with the
authors' published series.

Requirements: `pip install numpy pandas statsmodels matplotlib requests python-dotenv openpyxl`
and a `.env` with `FRED_API_KEY` / `BEA_API_KEY` (only needed for the data-fetch
cells; the BEA category tables and FRED controls must be reachable, i.e. run this
on a normal network).

## 0. Setup

In [ ]:
import os, csv, json, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, OUT, PROC, CFG = (REPO_ROOT/"data"/"raw"), REPO_ROOT/"outputs", REPO_ROOT/"data"/"processed", REPO_ROOT/"config"
for p in (RAW/"fred", RAW/"bea", OUT, PROC):
    p.mkdir(parents=True, exist_ok=True)
if load_dotenv: load_dotenv(REPO_ROOT/".env")
FRED_API_KEY, BEA_API_KEY = os.environ.get("FRED_API_KEY"), os.environ.get("BEA_API_KEY")
print("repo:", REPO_ROOT, "| FRED key:", bool(FRED_API_KEY), "| BEA key:", bool(BEA_API_KEY))

## 1. Benchmark model & residuals (Eqs. 1-3)

$$\pi_{i,t}=\mu_i+\rho_i\pi_{i,t-1}+\varepsilon_{i,t}\quad(3)$$
estimated on a 120-month rolling window per category; we keep the residual at the
window end. Baseline AR(1), window 120, run length k=3.

In [ ]:
AR_ORDER, WINDOW, RUN_LENGTH = 1, 120, 3

def ar_design(y, p):
    n = len(y); Y = y[p:]
    cols = [np.ones(n-p)] + [y[p-lag:n-lag] for lag in range(1, p+1)]
    return np.column_stack(cols), Y

def rolling_ar_residuals(inflation, ar_order=AR_ORDER, window=WINDOW):
    y = inflation.to_numpy(float); n = len(y); resid = np.full(n, np.nan)
    for end in range(window-1, n):
        yw = y[end-window+1:end+1]
        if not np.all(np.isfinite(yw)): continue
        X, Y = ar_design(yw, ar_order)
        beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
        resid[end] = (Y - X @ beta)[-1]
    return pd.Series(resid, index=inflation.index, name=inflation.name)

def residual_panel(panel, ar_order=AR_ORDER, window=WINDOW):
    return panel.apply(lambda c: rolling_ar_residuals(c, ar_order, window))

## 2. Momentum signals (Eqs. 4-5)
$$M^{+}_{i,t}=\prod_{k=0}^{2}\mathbf 1(\varepsilon_{i,t-k}>0),\quad
M^{-}_{i,t}=\prod_{k=0}^{2}\mathbf 1(\varepsilon_{i,t-k}<0)$$

In [ ]:
def momentum_signals(resid, run_length=RUN_LENGTH):
    pos = (resid > 0).astype(float); neg = (resid < 0).astype(float)
    mp = pos.rolling(run_length, min_periods=run_length).apply(np.prod, raw=True).fillna(0.0)
    mn = neg.rolling(run_length, min_periods=run_length).apply(np.prod, raw=True).fillna(0.0)
    return mp, mn

## 3. Shares & index (Eqs. 6-8)
$$S^{+}_t=\sum_i\omega_{i,t}M^{+}_{i,t},\;\; S^{-}_t=\sum_i\omega_{i,t}M^{-}_{i,t},\;\; ISM_t=S^{+}_t-S^{-}_t$$

In [ ]:
def expenditure_weighted_shares(mp, mn, weights):
    w = weights.reindex(columns=mp.columns)
    valid = mp.notna() & mn.notna() & w.notna()
    wn = w.where(valid).div(w.where(valid).sum(axis=1).replace(0, np.nan), axis=0)
    return (wn*mp).sum(axis=1).rename("S_pos"), (wn*mn).sum(axis=1).rename("S_neg")

def compute_ism(panel, weights, ar_order=AR_ORDER, window=WINDOW, run_length=RUN_LENGTH):
    resid = residual_panel(panel, ar_order, window)
    mp, mn = momentum_signals(resid, run_length)
    sp, sn = expenditure_weighted_shares(mp, mn, weights)
    return {"ism": (sp-sn).rename("ISM"), "s_pos": sp, "s_neg": sn, "residuals": resid}

### Engine self-test (no data)

In [ ]:
_r = pd.DataFrame({"A":[1,1,1,-1,1,1,1],"B":[-1,-1,-1,-1,-1,1,-1]}, dtype=float)
_mp,_mn = momentum_signals(_r,3)
assert _mp["A"].tolist()==[0,0,1,0,0,0,1] and _mn["B"].tolist()==[0,0,1,1,1,0,0]
print("engine self-test passed (Eqs. 4-5).")

## 4. Inflation transforms

In [ ]:
def monthly_inflation(price): return 100*(np.log(price)-np.log(price.shift(1)))
def yoy_inflation(price):     return 100*(np.log(price)-np.log(price.shift(12)))
def threemonth_inflation(p):  return 100*(np.log(p)-np.log(p.shift(3)))*(12/3)
def yoy_growth(level):        return 100*(level/level.shift(12)-1)

## 5. Data access (FRED + BEA, cached)

BEA tables 2.4.4U (`U20404`, price) and 2.4.5U (`U20405`, nominal) provide the
category data; FRED provides the controls. Each pull is cached under `data/raw/`.

In [ ]:
import sys as _sys, time as _time
_sys.path.insert(0, str(REPO_ROOT / "src"))

# Prefer the library's battle-tested client (retries 429/5xx with backoff,
# honors Retry-After, caches with provenance). Fall back to an inline robust
# fetcher if src/ism isn't importable.
try:
    from ism.datasources import FredClient as _FredClient, BeaClient as _BeaClient
    _fc, _bc = _FredClient(), _BeaClient()
    _USE_LIB = True
except Exception as _e:
    print("ism.datasources not importable; using inline fetchers:", _e)
    _USE_LIB = False

def _get_with_retry(url, params, timeout=60, retries=7):
    import requests
    last = None
    for k in range(retries):
        r = requests.get(url, params=params, timeout=timeout); last = r
        if r.status_code == 429 or 500 <= r.status_code < 600:
            wait = 0.0
            try: wait = float(r.headers.get("Retry-After", 0))
            except Exception: pass
            wait = wait or min(90.0, 2.0 * (2 ** k))
            print(f"  rate-limited ({r.status_code}); waiting {wait:.0f}s..."); _time.sleep(wait); continue
        r.raise_for_status(); return r
    last.raise_for_status(); return last

def fred_series(sid, force=False):
    if _USE_LIB:
        s = _fc.series(sid, force=force)
        s.index = pd.to_datetime(s.index).to_period("M").to_timestamp()
        return s.rename(sid)
    cache = RAW/"fred"/f"{sid}.json"
    if cache.exists() and not force:
        payload = json.loads(cache.read_text())
    else:
        if not FRED_API_KEY: raise RuntimeError("set FRED_API_KEY in .env")
        r = _get_with_retry("https://api.stlouisfed.org/fred/series/observations",
                            dict(series_id=sid, api_key=FRED_API_KEY, file_type="json"))
        payload = r.json(); cache.write_text(json.dumps(payload)); _time.sleep(0.4)
    obs = payload["observations"]; idx = pd.to_datetime([o["date"] for o in obs])
    val = pd.to_numeric(pd.Series([o["value"] for o in obs]).replace(".", pd.NA), errors="coerce")
    s = pd.Series(val.to_numpy(), index=idx, name=sid); s.index = s.index.to_period("M").to_timestamp()
    return s

def bea_table(table, force=False):
    if _USE_LIB:
        df = _bc.table(table, force=force)
    else:
        cache = RAW/"bea"/f"{table}_M.json"
        if cache.exists() and not force:
            payload = json.loads(cache.read_text())
        else:
            if not BEA_API_KEY: raise RuntimeError("set BEA_API_KEY in .env")
            r = _get_with_retry("https://apps.bea.gov/api/data/", dict(UserID=BEA_API_KEY,
                method="GetData", datasetname="NIUnderlyingDetail", TableName=table,
                Frequency="M", Year="ALL", ResultFormat="json"), timeout=120)
            payload = r.json(); cache.write_text(json.dumps(payload))
        df = pd.DataFrame(payload["BEAAPI"]["Results"]["Data"])
        df["DataValue"] = pd.to_numeric(df["DataValue"].astype(str).str.replace(",","",regex=False), errors="coerce")
    if "date" not in df.columns:
        df["date"] = pd.to_datetime(df["TimePeriod"].astype(str).str.replace("M","-")+"-01", errors="coerce")
    return df

## 6. Build the category panel from the pinned set

Category set = the 130 fourth-level categories in `config/pce_categories.csv`
(produced by `scripts/finalize_categories.py`). Price codes (`...RG`/`IA...`)
are matched to nominal (`...RC`/`LA...`) on a normalized key.

In [ ]:
def norm_key(c):
    c = str(c); return c[1:] if c[:2] in ("IA","LA") else c[:-1]

def build_category_panel():
    keys = pd.read_csv(CFG/"pce_categories.csv")["key"].astype(str).tolist()
    price, nominal = bea_table("U20404"), bea_table("U20405")
    for df in (price, nominal): df["key"] = df["SeriesCode"].map(norm_key)
    pw = price.pivot_table(index="date", columns="key", values="DataValue", aggfunc="first").sort_index()
    nw = nominal.pivot_table(index="date", columns="key", values="DataValue", aggfunc="first").sort_index()
    keys = [k for k in keys if k in pw.columns and k in nw.columns]
    infl = monthly_inflation(pw[keys])
    weights = nw[keys].div(nw[keys].sum(axis=1).replace(0, np.nan), axis=0)
    common = infl.index.intersection(weights.index)
    print(f"panel: {len(common)} months x {len(keys)} categories")
    return infl.loc[common], weights.loc[common]

inflation_panel, weights = build_category_panel()
res = compute_ism(inflation_panel, weights)
pd.concat([res["ism"], res["s_pos"], res["s_neg"]], axis=1).to_csv(PROC/"ism_index_replicated.csv")
res["ism"].dropna().tail()

## 7. Convergence vs the authors' published series

In [ ]:
def load_author():
    a = pd.read_excel(RAW/"ISM_public_author.xlsx"); a.columns=[c.strip() for c in a.columns]
    idx = pd.to_datetime(a["time_month"].astype(str).str.strip().str.replace("m","-")+"-01")
    return pd.DataFrame({"ISM":a["ISM Index"].values,"S_pos":a["Positive Momentum Component"].values,
                         "S_neg":a["Negative Momentum Component"].values}, index=idx)

truth = load_author()
def stat(x,y):
    j=pd.concat([x.rename("x"),y.rename("y")],axis=1).dropna(); d=j["x"]-j["y"]
    return dict(corr=round(j["x"].corr(j["y"]),4), rmse=round(float(np.sqrt((d**2).mean())),4),
                max_abs=round(float(d.abs().max()),4), n=len(j))
pd.DataFrame({"ISM":stat(res["ism"],truth["ISM"]), "S_pos":stat(res["s_pos"],truth["S_pos"]),
              "S_neg":stat(res["s_neg"],truth["S_neg"])}).T

In [ ]:
fig, ax = plt.subplots(figsize=(11,4.5))
ax.plot(truth.index, truth["ISM"], color="black", lw=1.6, label="Author ISM (published)")
ax.plot(res["ism"].index, res["ism"].values, color="tab:red", lw=1.0, alpha=.8, label="Replicated ISM")
ax.axhline(0, color="grey", lw=.6); ax.legend(); ax.set_title("ISM: replication vs author ground truth")
fig.tight_layout(); fig.savefig(OUT/"convergence_ISM.png", dpi=130); plt.show()

## 8. Figure 1 — ISM index, 12-month PCE inflation, and components

In [ ]:
pcepi = fred_series("PCEPI"); pce_yoy = yoy_inflation(pcepi).rename("pce_yoy")
fig,(axA,axB)=plt.subplots(2,1,figsize=(11,8),sharex=True)
axA.plot(res["ism"].index,res["ism"],color="tab:blue",lw=1.3,label="ISM Index"); axA.axhline(0,color="grey",lw=.6)
axA.set_ylabel("ISM Index"); ax2=axA.twinx()
ax2.plot(pce_yoy.index,pce_yoy,color="tab:orange",lw=1.0,label="PCE inflation (12m)"); ax2.set_ylabel("PCE inflation (12m, %)")
axA.set_title("Panel A: ISM index and 12-month PCE inflation"); axA.legend(loc="upper left")
axB.plot(res["s_pos"].index,res["s_pos"],color="tab:orange",lw=1.1,label="Positive share $S^+$")
axB.plot(res["s_neg"].index,res["s_neg"],color="tab:blue",lw=1.1,label="Negative share $S^-$")
axB.set_title("Panel B: components"); axB.legend(loc="upper left")
fig.tight_layout(); fig.savefig(OUT/"figure1_ISM.png",dpi=130); plt.show()

## 9. Controls for the forecasting regressions

FRED controls, plus two long-history series the paper uses that FRED doesn't fully provide:

- **V/U ratio**: JOLTS openings (`JTSJOL`, 2000+) spliced to **Barnichon (2010)** for the earlier sample (paper fn 15). Drop `data/raw/external/barnichon_hwi.csv` (2 cols: date, index) to enable the full-history splice; otherwise V/U starts 2000.
- **S&P 500 level**: **Shiller** `ie_data.xls` (monthly, auto-downloaded) for full history, since FRED's `SP500` is licence-limited (~10y).

With both present, Table 1's with-controls columns run on the full 1969+ sample.

In [ ]:
import sys as _sys
_sys.path.insert(0, str(REPO_ROOT / "src"))
try:
    from ism.external_data import load_shiller_sp500, load_barnichon_hwi, build_vu_ratio
    _HAVE_EXT = True
except Exception as _e:
    print("external_data not importable, falling back to FRED-only controls:", _e)
    _HAVE_EXT = False

def build_controls():
    pcepi = fred_series("PCEPI")
    pce_yoy = yoy_inflation(pcepi).rename("pce_yoy")
    pce_3m  = threemonth_inflation(pcepi).rename("pce_3m")
    infl_exp = fred_series("MICH").rename("infl_exp_1y").combine_first(pce_yoy.rename("infl_exp_1y"))
    oil = fred_series("WTISPLC").rename("oil_wti")
    rdpi_yoy = yoy_growth(fred_series("DSPIC96")).rename("rdpi_yoy")
    spread = (fred_series("GS10") - fred_series("FEDFUNDS")).rename("spread_10y_ffr")
    rec = fred_series("USREC").rename("nber_recession")

    # V/U ratio: splice Barnichon (if provided) to JOLTS over the unemployment level
    jolts, unemp = fred_series("JTSJOL"), fred_series("UNEMPLOY")
    barn = None
    if _HAVE_EXT:
        try: barn = load_barnichon_hwi()
        except Exception as e: print("V/U: Barnichon file not found -> JOLTS-only from 2000:", e)
        vu = build_vu_ratio(jolts, unemp, barn).rename("vu_ratio")
    else:
        vu = (jolts / unemp).rename("vu_ratio")

    # S&P 500 level: Shiller long history if available, else FRED SP500 (short)
    sp500 = None
    if _HAVE_EXT:
        try: sp500 = load_shiller_sp500().rename("sp500")
        except Exception as e: print("S&P: Shiller not available -> trying FRED SP500:", e)
    if sp500 is None:
        try: sp500 = fred_series("SP500").rename("sp500")
        except Exception: sp500 = pd.Series(dtype=float, name="sp500")

    out = pd.concat([pce_yoy, pce_3m, infl_exp, vu, oil, sp500, rdpi_yoy, spread, rec], axis=1)
    out.index = pd.to_datetime(out.index).to_period("M").to_timestamp()
    return out

controls = build_controls(); controls.tail()

## 10. Table 1 — in-sample forecasts

Dependent variable: 12-month PCE inflation $h$ months ahead. Regressors: constant
+ current 12m PCE inflation, then either the ISM index or its components, with/
without the control block. HC1 robust SEs.

In [ ]:
import statsmodels.api as sm
CONTROLS = ["pce_3m","infl_exp_1y","vu_ratio","oil_wti","sp500","rdpi_yoy","spread_10y_ffr","nber_recession"]

def insample(pce_yoy, horizon, momentum="ism", use_controls=False):
    y = pce_yoy.shift(-horizon).rename("y")
    X = pd.DataFrame({"pce_yoy": pce_yoy})
    if momentum=="ism":         X["ISM"]=res["ism"]
    elif momentum=="components":X["S_pos"]=res["s_pos"]; X["S_neg"]=res["s_neg"]
    if use_controls:
        for c in CONTROLS:
            if c in controls.columns: X[c]=controls[c]
    d = pd.concat([y,X],axis=1).dropna()
    m = sm.OLS(d["y"], sm.add_constant(d[X.columns])).fit(cov_type="HC1")
    def cs(n): return f"{m.params[n]:.3f} ({m.bse[n]:.3f})" if n in m.params else ""
    return dict(spec=f"{momentum}{'+ctrl' if use_controls else ''}", **{k:cs(k) for k in ["pce_yoy","ISM","S_pos","S_neg"]},
                adjR2=round(m.rsquared_adj,3), N=int(m.nobs))

rows=[]
for h in (12,24):
    for mom in ("none","components","ism"):
        for uc in (False, True):
            rows.append({"h":h, **insample(pce_yoy, h, mom, uc)})
pd.DataFrame(rows)

## 11. Feeding in your own forecasts

The engine consumes a plain `(inflation_panel, weights)`. To stress-test a
scenario, append projected future months and recompute:

```python
panel_ext = pd.concat([inflation_panel, future_infl])     # future_infl: months x same categories
wts_ext   = pd.concat([weights, future_weights])
res_ext   = compute_ism(panel_ext, wts_ext)
res_ext["ism"].tail(len(future_infl))                     # ISM over your forecast horizon
```

The same append pattern works for `controls`, so the Table-1 regressions can be
re-run under alternative macro scenarios.

## Sources & notes
All series documented in `config/sources.yaml`; category set pinned in
`config/pce_categories.csv`; convergence + methodology in
`docs/differences_report.md`. Replicated ISM correlates 0.99 with the authors'
series.